# Insurance Premium Billing & Collections — Data Generation

This notebook generates a simulated insurance premium billing dataset used to build a
reconciliation and collections aging analysis. It creates four related tables:
**agencies**, **policies**, **invoices**, and **payments** — then merges them into a
single reconciliation view showing which invoices are paid, past due, and by how much.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

## Agencies

10 insurance agencies, each assigned a payment behavior profile
(`reliable`, `slow`, `problem`) that will later drive how likely their
invoices are to be paid on time, late, or short.

In [3]:
agencies = pd.DataFrame({
    "agency_id": [f"A{i:02d}" for i in range(1, 11)],
    "agency_name": ["Summit Insurance Group", "Harbor Risk Advisors",
                     "Liberty Insurance Agency", "Keystone Benefits & Risk",
                     "Pioneer Underwriters", "Cardinal Insurance Group",
                     "Bayview Risk Advisors", "Ironwood Insurance Agency",
                     "Sterling Benefits & Risk", "Northgate Underwriters"],
    "payment_profile": ["reliable"]*5 + ["slow"]*3 + ["problem"]*2
})
agencies.head()

,agency_id,agency_name,payment_profile
0,A01,Summit Insurance Group,reliable
1,A02,Harbor Risk Advisors,reliable
2,A03,Liberty Insurance Agency,reliable
3,A04,Keystone Benefits & Risk,reliable
4,A05,Pioneer Underwriters,reliable


## Policies

50 insurance policies, spread randomly across the 10 agencies, covering
three lines of business (Commercial Auto, General Liability, Commercial
Property) with a randomly assigned annual premium.

In [4]:
n_policies = 50
policies = pd.DataFrame({
    "policy_number": [f"POL-{1000+i}" for i in range(n_policies)],
    "agency_id": rng.choice(agencies["agency_id"], size=n_policies),
    "line_of_business": rng.choice(
        ["Commercial Auto", "General Liability", "Commercial Property"],
        size=n_policies),
    "annual_premium": np.round(rng.uniform(6000, 60000, n_policies), 0)
})
policies.head()

,policy_number,agency_id,line_of_business,annual_premium
0,POL-1000,A01,Commercial Property,16795.0
1,POL-1001,A08,Commercial Auto,6398.0
2,POL-1002,A07,General Liability,48494.0
3,POL-1003,A05,General Liability,41902.0
4,POL-1004,A05,General Liability,44079.0


## Invoices

Each policy is billed monthly for three months (June–August 2026).
The `premium_billed` amount is the annual premium divided evenly across
12 months, and each invoice's `due_date` is 30 days after it's issued.

In [5]:
invoice_dates = pd.date_range("2026-06-01", "2026-08-01", freq="MS")
rows = []
for p in policies.itertuples():
    for inv_date in invoice_dates:
        rows.append({
            "policy_number": p.policy_number,
            "agency_id": p.agency_id,
            "invoice_date": inv_date,
            "premium_billed": round(p.annual_premium / 12, 2)
        })
invoices = pd.DataFrame(rows)
invoices["due_date"] = invoices["invoice_date"] + pd.Timedelta(days=30)
invoices.insert(0, "invoice_id", [f"INV-{i:04d}" for i in range(1, len(invoices)+1)])
invoices.head()

,invoice_id,policy_number,agency_id,invoice_date,premium_billed,due_date
0,INV-0001,POL-1000,A01,2026-06-01,1399.58,2026-07-01
1,INV-0002,POL-1000,A01,2026-07-01,1399.58,2026-07-31
2,INV-0003,POL-1000,A01,2026-08-01,1399.58,2026-08-31
3,INV-0004,POL-1001,A08,2026-06-01,533.17,2026-07-01
4,INV-0005,POL-1001,A08,2026-07-01,533.17,2026-07-31


## Payments

Each invoice is assigned a payment scenario — **on_time**, **late**, or
**short** — based on its agency's payment profile. Reliable agencies pay
on time most often; problem agencies are far more likely to pay late or
short. The `as_of` date caps how far into the future we simulate, so only
payments that would realistically have arrived by September 15, 2026 are
included.

In [6]:
as_of = pd.Timestamp("2026-09-15")
profile_map = agencies.set_index("agency_id")["payment_profile"].to_dict()
invoices["profile"] = invoices["agency_id"].map(profile_map)

scenario_probs = {
    "reliable": [0.85, 0.10, 0.05],   # on_time, late, short
    "slow":     [0.40, 0.50, 0.10],
    "problem":  [0.20, 0.40, 0.40]
}
scenarios = ["on_time", "late", "short"]
invoices["scenario"] = [
    rng.choice(scenarios, p=scenario_probs[prof]) for prof in invoices["profile"]
]

pay_rows = []
for inv in invoices.itertuples():
    if inv.scenario == "on_time":
        pay_date = inv.due_date - pd.Timedelta(days=int(rng.integers(0, 5)))
        amount = inv.premium_billed
    elif inv.scenario == "late":
        pay_date = inv.due_date + pd.Timedelta(days=int(rng.integers(5, 60)))
        amount = inv.premium_billed
    else:  # short
        pay_date = inv.due_date + pd.Timedelta(days=int(rng.integers(0, 20)))
        amount = round(inv.premium_billed * rng.uniform(0.6, 0.9), 2)

    if pay_date <= as_of:
        pay_rows.append({
            "invoice_id": inv.invoice_id,
            "agency_id": inv.agency_id,
            "payment_date": pay_date,
            "payment_amount": amount
        })

payments = pd.DataFrame(pay_rows)
payments.insert(0, "payment_id", [f"PAY-{i:04d}" for i in range(1, len(payments)+1)])
payments.head()

,payment_id,invoice_id,agency_id,payment_date,payment_amount
0,PAY-0001,INV-0001,A01,2026-07-12,1399.58
1,PAY-0002,INV-0002,A01,2026-07-28,1399.58
2,PAY-0003,INV-0003,A01,2026-08-27,1399.58
3,PAY-0004,INV-0004,A08,2026-07-18,392.50
4,PAY-0005,INV-0005,A08,2026-09-03,533.17


##  Reconciliation

This is the core output: for every invoice, compare what was **billed**
against what was actually **paid**, calculate the remaining **balance**,
and classify each invoice's **status** (Paid, Current, or Past Due).
Past-due invoices are further bucketed into aging ranges (1-30, 31-60,
60+ days) — the same categories used in real accounts receivable aging
reports.

In [14]:
recon = invoices.merge(
    payments.groupby("invoice_id")["payment_amount"].sum().reset_index(),
    on="invoice_id", how="left"
)
recon = recon.merge(agencies[["agency_id", "agency_name"]], on="agency_id", how="left")
recon = recon.merge(policies[["policy_number", "line_of_business"]], on="policy_number", how="left")
recon["payment_amount"] = recon["payment_amount"].fillna(0)
recon["balance"] = round(recon["premium_billed"] - recon["payment_amount"], 2)

recon["status"] = np.select(
    [recon["balance"] <= 0.01, recon["due_date"] >= as_of],
    ["Paid", "Current"],
    default="Past Due"
)

recon["days_past_due"] = (as_of - recon["due_date"]).dt.days
recon["aging_bucket"] = pd.cut(
    recon["days_past_due"],
    bins=[-9999, 0, 30, 60, 9999],
    labels=["Current", "1-30 Days", "31-60 Days", "60+ Days"]
)
recon["aging_bucket"] = recon["aging_bucket"].astype(str)
recon.loc[recon["status"] == "Paid", "aging_bucket"] = "Paid"
recon.head()

,invoice_id,policy_number,agency_id,invoice_date,premium_billed,due_date,profile,scenario,payment_amount,agency_name,line_of_business,balance,status,days_past_due,aging_bucket
0,INV-0001,POL-1000,A01,2026-06-01,1399.58,2026-07-01,reliable,late,1399.58,Summit Insurance Group,Commercial Property,0.00,Paid,76,Paid
1,INV-0002,POL-1000,A01,2026-07-01,1399.58,2026-07-31,reliable,on_time,1399.58,Summit Insurance Group,Commercial Property,0.00,Paid,46,Paid
2,INV-0003,POL-1000,A01,2026-08-01,1399.58,2026-08-31,reliable,on_time,1399.58,Summit Insurance Group,Commercial Property,0.00,Paid,15,Paid
3,INV-0004,POL-1001,A08,2026-06-01,533.17,2026-07-01,slow,short,392.50,Ironwood Insurance Agency,Commercial Auto,140.67,Past Due,76,60+ Days
4,INV-0005,POL-1001,A08,2026-07-01,533.17,2026-07-31,slow,late,533.17,Ironwood Insurance Agency,Commercial Auto,0.00,Paid,46,Paid


In [16]:
print(recon.columns.tolist())

['invoice_id', 'policy_number', 'agency_id', 'invoice_date', 'premium_billed', 'due_date', 'profile', 'scenario', 'payment_amount', 'agency_name', 'line_of_business', 'balance', 'status', 'days_past_due', 'aging_bucket']


##  Export

Save all four tables to CSV for use in the Excel reconciliation workbook
and the SQL/Python analysis notebooks.

In [18]:
agencies.to_csv("../data/raw/agencies.csv", index=False)
invoices.to_csv("../data/raw/invoices.csv", index=False)
payments.to_csv("../data/raw/payments.csv", index=False)
recon.to_csv("../data/raw/reconciliation.csv", index=False)

In [19]:
check = pd.read_csv("../data/raw/reconciliation.csv")
print(check.columns.tolist())

['invoice_id', 'policy_number', 'agency_id', 'invoice_date', 'premium_billed', 'due_date', 'profile', 'scenario', 'payment_amount', 'agency_name', 'line_of_business', 'balance', 'status', 'days_past_due', 'aging_bucket']


##  Quick Summary Check

In [9]:
print(recon["status"].value_counts())
print(recon.groupby("aging_bucket")["balance"].sum())

status
Paid        108
Past Due     42
Name: count, dtype: int64
aging_bucket
1-30 Days     40205.45
31-60 Days    15857.91
60+ Days       6448.03
Paid              0.00
Name: balance, dtype: float64
